# Entity Linking and Knowledge Representation

Owner: Member 3

This notebook maps extracted claim triples from Notebook 1 into a knowledge graph-compatible representation for downstream KG reasoning.

## Role in the Project

Notebook 1 produces extracted triples such as `subject`, `relation`, and `object`. This notebook links entity-like subjects and objects to Wikidata identifiers and maps selected relations to Wikidata property IDs when the mapping is defensible.

The output is `03_Entity_Linking_KR/linked_entities.json`, which is the expected input for `04_KG_Reasoning`.

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'LIAR_dataset').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

MODULE_DIR = PROJECT_ROOT / '03_Entity_Linking_KR'
sys.path.insert(0, str(MODULE_DIR))

from run_entity_linking import normalize_input_records, run_entity_linking

INPUT_PATH = PROJECT_ROOT / '01_Claim_Extraction' / 'extracted_triples_filtered.json'
OUTPUT_PATH = MODULE_DIR / 'linked_entities.json'
SUMMARY_PATH = MODULE_DIR / 'entity_linking_summary.json'

INPUT_PATH, OUTPUT_PATH, SUMMARY_PATH

(PosixPath('/Users/shanusharma/AIaP-Group_Project/01_Claim_Extraction/extracted_triples_filtered.json'),
 PosixPath('/Users/shanusharma/AIaP-Group_Project/03_Entity_Linking_KR/linked_entities.json'),
 PosixPath('/Users/shanusharma/AIaP-Group_Project/03_Entity_Linking_KR/entity_linking_summary.json'))

## Input Schema

The preferred input is `01_Claim_Extraction/extracted_triples_filtered.json`. If Notebook 1 does not provide `claim_id`, this module creates stable IDs such as `claim-00001`. If Notebook 1 uses `confidence`, this module normalizes it to `extraction_confidence`.

In [2]:
records = json.loads(INPUT_PATH.read_text())
df = normalize_input_records(records)
df.head()

,claim_id,raw_claim,label,speaker,subject,subject_type,relation,object,object_type,date,extraction_confidence,entities
0,claim-00001,Says the Annies List political group supports ...,false,dwayne-bohac,Annies List,ORG,say,demand,UNKNOWN,NaN,0.8949,"[{""word"": "" Annies List"", ""type"": ""ORG"", ""scor..."
1,claim-00002,When did the decline of coal start? It started...,half-true,scott-surovell,the decline,UNKNOWN,do,George W.) Bush,PER,NaN,0.8983,"[{""word"": "" George W.) Bush"", ""type"": ""PER"", ""..."
2,claim-00003,"Hillary Clinton agrees with John McCain ""by vo...",mostly-true,barack-obama,Hillary Clinton,PER,agree,John McCain,PER,NaN,0.9998,"[{""word"": "" Hillary Clinton"", ""type"": ""PER"", ""..."
3,claim-00004,Health care reform legislation is likely to ma...,false,blog-posting,Health care reform legislation,UNKNOWN,be,free sex change surgeries,UNKNOWN,NaN,0.3000,[]
4,claim-00005,The economic turnaround started at the end of ...,half-true,charlie-crist,The economic turnaround,UNKNOWN,start,at the end,UNKNOWN,NaN,0.3000,[]


## Method

The entity-linking method uses the following safeguards:

- link only entity-like values: people, organisations, locations, geopolitical entities, and miscellaneous named entities
- keep dates, percentages, counts, and money values as literal objects
- expand common ambiguous political names before search, such as `Obama -> Barack Obama`, `McCain -> John McCain`, and `ISIS -> Islamic State`
- reject obvious bad Wikidata hits such as family-name pages, given-name pages, disambiguation pages, and unrelated academic journals
- map `property_id` only when the relation or claim pattern gives a defensible Wikidata property

In [3]:
# Full run. This calls the Wikidata API for entity search.
# For a fast smoke test without network calls, add: offline=True, max_records=20
results = run_entity_linking(PROJECT_ROOT, sleep_seconds=0.05)
summary = results['summary']
summary

Linking entities: 100%|██████████| 500/500 [02:10<00:00,  3.83it/s]


{'input_path': '01_Claim_Extraction/extracted_triples_filtered.json',
 'output_path': '03_Entity_Linking_KR/linked_entities.json',
 'total_claims': 500,
 'subjects_linked': 243,
 'objects_linked': 83,
 'records_with_any_entity_link': 276,
 'relations_with_property_id': 1,
 'average_linking_confidence': 0.374,
 'low_confidence_records_below_0_4': 250,
 'manual_subject_links': 70,
 'manual_object_links': 21,
 'top_relations': [{'relation': 'say', 'count': 99},
  {'relation': 'be', 'count': 72},
  {'relation': 'have', 'count': 26},
  {'relation': 'support', 'count': 8},
  {'relation': 'vote', 'count': 8},
  {'relation': 'take', 'count': 5},
  {'relation': 'cut', 'count': 5},
  {'relation': 'go', 'count': 5},
  {'relation': 'on', 'count': 4},
  {'relation': 'have be', 'count': 4},
  {'relation': 'want', 'count': 4},
  {'relation': 'try', 'count': 4},
  {'relation': 'pay', 'count': 4},
  {'relation': 'offer', 'count': 3},
  {'relation': 'make', 'count': 3}]}

## Linked Output Preview

Each output row preserves the original claim fields and adds Wikidata entity IDs, labels, descriptions, relation property metadata, confidence, and notes.

In [4]:
linked_df = results['linked_df']
linked_df.head()

,claim_id,raw_claim,label,speaker,subject,subject_type,subject_kg_id,subject_kg_label,subject_kg_description,subject_link_source,...,object,object_type,object_kg_id,object_kg_label,object_kg_description,object_link_source,date,extraction_confidence,linking_confidence,linking_notes
0,claim-00001,Says the Annies List political group supports ...,false,dwayne-bohac,Annies List,ORG,NaN,NaN,NaN,NaN,...,demand,NaN,NaN,NaN,NaN,NaN,NaN,0.8949,0.224,Subject not linked or not suitable for linking...
1,claim-00002,When did the decline of coal start? It started...,half-true,scott-surovell,the decline,NaN,NaN,NaN,NaN,NaN,...,George W.) Bush,PER,Q207,George W. Bush,president of the United States from 2001 to 2009,manual_alias,NaN,0.8983,0.421,Subject not linked or not suitable for linking...
2,claim-00003,"Hillary Clinton agrees with John McCain ""by vo...",mostly-true,barack-obama,Hillary Clinton,PER,Q6294,Hillary Clinton,American politician and diplomat,manual_alias,...,John McCain,PER,Q10390,John McCain,American politician,manual_alias,NaN,0.9998,0.789,Subject linked via manual_alias using query 'H...
3,claim-00004,Health care reform legislation is likely to ma...,false,blog-posting,Health care reform legislation,NaN,NaN,NaN,NaN,NaN,...,free sex change surgeries,NaN,NaN,NaN,NaN,NaN,NaN,0.3000,0.075,Subject not linked or not suitable for linking...
4,claim-00005,The economic turnaround started at the end of ...,half-true,charlie-crist,The economic turnaround,NaN,NaN,NaN,NaN,NaN,...,at the end,NaN,NaN,NaN,NaN,NaN,NaN,0.3000,0.075,Subject not linked or not suitable for linking...


## Handoff Validation

The fields below are required by the shared project schema and should be present for every record before moving to KG reasoning.

In [5]:
required_fields = [
    'claim_id',
    'raw_claim',
    'subject',
    'subject_kg_id',
    'object',
    'object_kg_id',
    'relation',
    'property_id',
    'linking_confidence',
    'linking_notes',
]

missing = {field: int(linked_df[field].isna().sum()) if field in linked_df else 'missing column' for field in required_fields}
missing

{'claim_id': 0,
 'raw_claim': 0,
 'subject': 0,
 'subject_kg_id': 257,
 'object': 4,
 'object_kg_id': 417,
 'relation': 0,
 'property_id': 499,
 'linking_confidence': 0,
 'linking_notes': 0}

## Results for Report

Use `entity_linking_summary.json` for concise report metrics such as number of linked subjects, linked objects, mapped properties, low-confidence records, and average linking confidence.

In [6]:
json.loads(SUMMARY_PATH.read_text())

{'input_path': '01_Claim_Extraction/extracted_triples_filtered.json',
 'output_path': '03_Entity_Linking_KR/linked_entities.json',
 'total_claims': 500,
 'subjects_linked': 243,
 'objects_linked': 83,
 'records_with_any_entity_link': 276,
 'relations_with_property_id': 1,
 'average_linking_confidence': 0.374,
 'low_confidence_records_below_0_4': 250,
 'manual_subject_links': 70,
 'manual_object_links': 21,
 'top_relations': [{'relation': 'say', 'count': 99},
  {'relation': 'be', 'count': 72},
  {'relation': 'have', 'count': 26},
  {'relation': 'support', 'count': 8},
  {'relation': 'vote', 'count': 8},
  {'relation': 'take', 'count': 5},
  {'relation': 'cut', 'count': 5},
  {'relation': 'go', 'count': 5},
  {'relation': 'on', 'count': 4},
  {'relation': 'have be', 'count': 4},
  {'relation': 'want', 'count': 4},
  {'relation': 'try', 'count': 4},
  {'relation': 'pay', 'count': 4},
  {'relation': 'offer', 'count': 3},
  {'relation': 'make', 'count': 3}]}

## Limitations

Entity linking remains uncertain. Some extracted triples contain vague subjects or objects, such as pronouns, generic noun phrases, or incomplete names. Those records are kept with low confidence instead of being removed, because downstream reasoning should know when evidence coverage is weak. Generic relations such as `say`, `be`, and `have` are not forced into Wikidata properties unless the claim pattern is specific enough.